# Exercise 2.2: A Small CNN for MNIST with PyTorch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/exercise_2_2_mnist_cnn_pytorch.ipynb)

This exercise introduces a small convolutional neural network (CNN) using PyTorch. The dataset is MNIST: 28 x 28 grayscale images of handwritten digits.

You will practice:

- loading an image dataset with `torchvision`
- inspecting image batches
- building a small CNN with `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d`, and `nn.Linear`
- training with `CrossEntropyLoss` and `Adam`
- tracking train and validation performance
- evaluating the model with accuracy, a confusion matrix, and example predictions

The mathematics are intentionally light. Focus on the workflow: data loader, model, loss, backpropagation, optimizer update, and evaluation.


## 0. Colab setup

Run this first in Colab. The MNIST dataset will be downloaded automatically by `torchvision`.


In [ ]:
!pip -q install torch torchvision scikit-learn pandas matplotlib seaborn

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from sklearn.metrics import ConfusionMatrixDisplay, classification_report

sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


## 1. Load MNIST

MNIST contains handwritten digits from 0 to 9. Each image has one channel and a size of 28 x 28 pixels.

For a faster classroom exercise, we use the full test set but only a subset of the training set. You can increase `TRAIN_SUBSET_SIZE` if you want better accuracy.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

DATA_DIR = Path("data")
full_train = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

TRAIN_SUBSET_SIZE = 12000
VAL_SIZE = 2000
TRAIN_SIZE = TRAIN_SUBSET_SIZE - VAL_SIZE

small_train, _ = random_split(
    full_train,
    [TRAIN_SUBSET_SIZE, len(full_train) - TRAIN_SUBSET_SIZE],
    generator=torch.Generator().manual_seed(SEED),
)
train_dataset, val_dataset = random_split(
    small_train,
    [TRAIN_SIZE, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

print("Train / val / test:", len(train_dataset), len(val_dataset), len(test_dataset))
xb, yb = next(iter(train_loader))
print("Image batch:", xb.shape)
print("Label batch:", yb.shape)


## 2. Inspect a batch of images

The tensor shape is `[batch_size, channels, height, width]`. For MNIST, channels is `1` because the images are grayscale.


In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(3, 6, figsize=(10, 5))
axes = axes.ravel()
for ax, image, label in zip(axes, images[:18], labels[:18]):
    ax.imshow(image.squeeze(0), cmap="gray")
    ax.set_title(f"label: {label.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()


### Student task

Look at the batch and answer briefly:

- Which digits look visually similar?
- Which examples might be difficult even for a human?
- Why is an image model different from the tabular model in Exercise 2.1?


## 3. Define a small CNN

A CNN learns small filters that slide over the image. Early filters often detect simple visual patterns such as edges and corners. Later layers combine these patterns into digit-specific evidence.

This model is intentionally small so that it trains quickly in Colab.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SmallCNN().to(DEVICE)
print(model)


In [ ]:
with torch.no_grad():
    sample_logits = model(images[:4].to(DEVICE))

print("Input shape: ", images[:4].shape)
print("Output shape:", sample_logits.shape)
print("One row of logits:", sample_logits[0].cpu().numpy().round(3))


## 4. Train the CNN

The training steps are the same as in Exercise 2.1:

1. compute logits with a forward pass
2. compute the loss
3. run `loss.backward()`
4. update parameters with `optimizer.step()`

The difference is the model architecture: CNN layers are designed for images.


In [ ]:
def evaluate(model, data_loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)

            total_loss += loss.item() * len(images)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += len(images)

    return total_loss / total, correct / total

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 4
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_total = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_total += loss.item() * len(images)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += len(images)

    train_loss = train_loss_total / train_total
    train_acc = train_correct / train_total
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
        }
    )

    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
        f"train acc {train_acc:.3f} | val acc {val_acc:.3f}"
    )

history_df = pd.DataFrame(history)


## 5. Plot learning curves

The validation curve helps you see whether the model is still improving or starting to overfit.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(data=history_df, x="epoch", y="train_loss", marker="o", label="train", ax=axes[0])
sns.lineplot(data=history_df, x="epoch", y="val_loss", marker="o", label="validation", ax=axes[0])
axes[0].set_title("Loss")
axes[0].set_ylabel("Cross-entropy loss")

sns.lineplot(data=history_df, x="epoch", y="train_accuracy", marker="o", label="train", ax=axes[1])
sns.lineplot(data=history_df, x="epoch", y="val_accuracy", marker="o", label="validation", ax=axes[1])
axes[1].set_title("Accuracy")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.02)

plt.tight_layout()
plt.show()


### Student task

Change one setting and rerun training:

- `EPOCHS`: try `1`, `4`, and `8`
- `TRAIN_SUBSET_SIZE`: try `3000` and `12000`
- first convolution channels: change `16` to `8` or `32`
- learning rate: change `0.001` to `0.0003` or `0.003`

What improves speed? What improves accuracy?


## 6. Evaluate on the test set

The test set should be used at the end, after you have chosen your settings.


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"Test loss:     {test_loss:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(DEVICE))
        predictions = logits.argmax(dim=1).cpu()
        all_predictions.append(predictions)
        all_labels.append(labels)

y_pred = torch.cat(all_predictions).numpy()
y_true = torch.cat(all_labels).numpy()

print(classification_report(y_true, y_pred, digits=3))

fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("MNIST CNN confusion matrix")
plt.tight_layout()
plt.show()


## 7. Inspect correct and incorrect predictions

The most useful errors are often visual: they show where the model confuses similar-looking digits.


In [ ]:
test_images, test_labels = next(iter(DataLoader(test_dataset, batch_size=256, shuffle=True)))
model.eval()
with torch.no_grad():
    logits = model(test_images.to(DEVICE)).cpu()
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)

wrong = torch.where(preds != test_labels)[0]
right = torch.where(preds == test_labels)[0]

fig, axes = plt.subplots(2, 6, figsize=(11, 4.5))
for ax, idx in zip(axes[0], right[:6]):
    ax.imshow(test_images[idx].squeeze(0), cmap="gray")
    ax.set_title(f"true {test_labels[idx].item()} / pred {preds[idx].item()}")
    ax.axis("off")

for ax, idx in zip(axes[1], wrong[:6]):
    ax.imshow(test_images[idx].squeeze(0), cmap="gray")
    ax.set_title(f"true {test_labels[idx].item()} / pred {preds[idx].item()}")
    ax.axis("off")

axes[0, 0].set_ylabel("correct")
axes[1, 0].set_ylabel("wrong")
plt.tight_layout()
plt.show()


## 8. Short reflection

Answer briefly:

- What does `Conv2d` do that a tabular linear layer does not?
- Why do we use `MaxPool2d` in this small CNN?
- Which digits are most often confused?
- What would you try next to improve the model?

The goal is a working mental model of CNN training, not a perfect MNIST score.
